# A1 Pairs/OU — 03 · OU Fit + Cost-Realistic Backtest + Gauntlet

For each surviving pair: fit the OU spread (→ half-life), backtest a z-score rule on
**OOS** with **realistic taker costs on both legs** (cross-spread + STT + fees), then
run the same gauntlet we used in Recon (`basecamp_recon.arm_stats`): n_eff → PSR →
**DSR deflated by N_TRIALS** → PBO.

> The bar is unchanged: **DSR ≥ 0.95, PBO ≤ 0.20, net of cost, out-of-sample.**
> Anything else is a hypothesis, not an edge.

In [ ]:
import os, sys
import numpy as np, pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath("../.."))
from basecamp_recon.arm_stats import sharpe, n_eff, psr, deflated_sharpe, pbo_cscv

OUT = "out"
panel = pd.read_parquet(f"{OUT}/panel.parquet")
stats = pd.read_csv(f"{OUT}/symbol_stats.csv", index_col=0)
cand  = pd.read_csv(f"{OUT}/candidate_pairs.csv")
N_TRIALS = int(pd.read_csv(f"{OUT}/screen_meta.csv", index_col=0).loc["N_TRIALS"].iloc[0])
print(f"{len(cand)} candidate pairs | N_TRIALS={N_TRIALS}")

In [ ]:
# ── OU fit via AR(1) on the spread:  S_t+1 = c + b S_t + e ───────────────────────
BAR_MIN = 1.0    # minutes per bar (matches NB1 BAR_FREQ)

def ou_fit(spread):
    s = spread.dropna().values
    x = sm.add_constant(s[:-1]); y = s[1:]
    c, b = sm.OLS(y, x).fit().params
    b = min(max(b, 1e-6), 0.999999)
    theta = -np.log(b) / BAR_MIN                      # per-minute mean-reversion
    half_life = np.log(2)/theta if theta > 0 else np.inf   # minutes
    mu = c/(1-b)
    sigma_eq = np.std(y - (c + b*s[:-1])) / np.sqrt(1-b**2)
    return dict(theta=theta, half_life_min=half_life, mu=mu, sigma_eq=sigma_eq, b=b)

def spread_series(x, y):
    s = panel[[x, y]].dropna()
    beta = sm.OLS(np.log(s[x]).values, sm.add_constant(np.log(s[y]).values)).fit().params
    sp = np.log(s[x]) - (beta[0] + beta[1]*np.log(s[y]))
    return sp, beta[1], s

In [ ]:
# ── taker cost per round-trip for the PAIR (both legs), in spread-log units ──────
STT = 0.000125          # sell-side; applied ~once per leg round-trip (approx)
BROKER = 20.0           # per order

def leg_cost_bps(sym):
    px, sp = stats.loc[sym, "med_price"], stats.loc[sym, "med_spread"]
    cross_bps = (sp / px) * 1e4                       # cross the spread once (taker)
    stt_bps   = STT * 1e4                             # sell leg
    misc_bps  = (BROKER / (px * stats.loc[sym,"lot"])) * 1e4 * 2   # brokerage both orders
    return cross_bps + stt_bps + misc_bps

def pair_roundtrip_cost_logret(x, y):
    # round trip = open+close, both legs taker → ~2x leg cost per leg, summed over 2 legs
    return (leg_cost_bps(x) + leg_cost_bps(y)) / 1e4   # as a log-return-ish fraction
print({s: round(leg_cost_bps(s),2) for s in stats.index})

In [ ]:
# ── z-score backtest on OOS spread ──────────────────────────────────────────────
SPLIT = 0.6
ENTRY_Z, EXIT_Z, STOP_Z = 2.0, 0.5, 4.0
ZWIN = 120              # rolling window for z (~2h at 1min)

def backtest(x, y):
    sp, beta, s = spread_series(x, y)
    ou = ou_fit(sp)
    k = int(len(sp)*SPLIT); oos = sp.iloc[k:]
    z = (oos - oos.rolling(ZWIN).mean()) / oos.rolling(ZWIN).std()
    pos = pd.Series(0.0, index=oos.index)
    state = 0
    for t in range(1, len(oos)):
        zt = z.iloc[t]
        if np.isnan(zt): continue
        if state == 0 and abs(zt) > ENTRY_Z:   state = -np.sign(zt)   # fade the spread
        elif state != 0 and (abs(zt) < EXIT_Z or abs(zt) > STOP_Z): state = 0
        pos.iloc[t] = state
    dspread = oos.diff().fillna(0.0)
    gross = pos.shift(1).fillna(0.0) * dspread        # spread log-return pnl (beta-hedged)
    turns = pos.diff().abs().fillna(0.0)
    cost  = turns * (pair_roundtrip_cost_logret(x, y) / 2.0)   # cost per side-change
    net = gross - cost
    daily = net.groupby(net.index.normalize()).sum()
    return ou, daily, pos

results = {}
for _, r in cand.iterrows():
    ou, daily, pos = backtest(r.x, r.y)
    results[f"{r.x}-{r.y}"] = {"ou": ou, "daily": daily,
                               "n_trades": int(pos.diff().abs().sum()/2)}
{k: (round(v["ou"]["half_life_min"],1), v["n_trades"]) for k,v in results.items()}

In [ ]:
# ── the gauntlet, per pair (net-of-cost OOS daily returns) ───────────────────────
sr_trials = []
table = []
for name, v in results.items():
    d = v["daily"]; r = d.values
    if len(r) < 3 or r.std() == 0:
        continue
    sr = sharpe(r); sr_trials.append(sr)
for name, v in results.items():
    d = v["daily"]; r = d.values
    if len(r) < 3 or r.std() == 0:
        table.append({"pair": name, "note": "too few days / zero var"}); continue
    sr = sharpe(r)
    dsr, sr_star = deflated_sharpe(sr, len(r), float(pd.Series(r).skew()),
                                   float(pd.Series(r).kurt()+3.0),
                                   np.array(sr_trials if len(sr_trials)>1 else [sr,0.0]))
    table.append({"pair": name, "days": len(r), "half_life_min": round(v["ou"]["half_life_min"],1),
                  "trades": v["n_trades"], "net_total": round(float(r.sum()),5),
                  "sharpe_day": round(sr,2), "PSR>0": round(psr(sr,len(r),0,3),2),
                  "DSR": round(dsr,2)})
gauntlet = pd.DataFrame(table)
print("N_TRIALS used for deflation:", N_TRIALS, "(note: also deflate by sweep below)")
gauntlet

In [ ]:
# ── PBO across the candidate pairs (CSCV) ───────────────────────────────────────
M = pd.DataFrame({name: v["daily"] for name, v in results.items()
                  if len(v["daily"]) >= 4}).dropna()
if M.shape[1] >= 2 and M.shape[0] >= 4:
    print("PBO:", pbo_cscv(M, s=min(6, M.shape[0])))
else:
    print("not enough pairs/days for a meaningful PBO — treat single-pair results with extreme caution")

In [ ]:
# ── equity curves ───────────────────────────────────────────────────────────────
plt.figure(figsize=(13,5))
for name, v in results.items():
    v["daily"].cumsum().plot(label=name)
plt.legend(fontsize=8); plt.title("OOS cumulative net-of-cost spread P&L (log-units)"); plt.show()

## Verdict checklist (be ruthless)

- **DSR ≥ 0.95 and PBO ≤ 0.20**, net of cost, OOS — *and* remember DSR here is only
  deflated by `N_TRIALS` pairs; if you *also* swept ENTRY_Z/EXIT_Z/ZWIN, the true trial
  count is larger and DSR is optimistic. Don't sweep-and-pick.
- **Half-life sane** (minutes-to-hours, stable) — a 1-bar half-life is noise; a
  multi-day half-life can't be traded intraday on this capital.
- **Costs didn't do all the work** — check `net_total` vs a zero-cost run; if the edge
  is razor-thin above cost, it won't survive real fills (we learned that the hard way).

**If nothing clears the bar:** that's a clean, cheap kill of A1 — log it and move to the
next candidate in `SEAT_AND_STRATEGIES.md`. **If something does:** the *only* next step is
a queue-/cost-aware paper arm on live data — never straight to capital.